# SVTRAF — Notebook 1: Framework Definition

**Purpose:** Define the SVTRAF framework architecture: the vulnerability taxonomy and the composite scoring formula.

**Research context:** MSc Cybersecurity dissertation, National College of Ireland.
SVTRAF (Structured Vulnerability Taxonomy and Risk Assessment Framework) is a blockchain-specific
vulnerability scoring framework validated against CVSS v3.1 on 150 smart contracts.

**This notebook answers:**
1. What gap does SVTRAF fill?
2. What is the 4-category, 17-vulnerability taxonomy?
3. How is the composite scoring formula constructed and justified?

**Runtime:** ~2 minutes

In [ ]:
# Imports and configuration
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', None)

print('All imports successful')
print(f'Pandas: {pd.__version__} | NumPy: {np.__version__}')

## 1. Research Problem and Gap Analysis

Smart contract vulnerabilities differ from traditional software defects in three fundamental ways:

1. **Immutability** — deployed contracts cannot be patched after deployment.
2. **Autonomous execution** — exploits execute in single transactions without human intervention.
3. **Direct financial harm** — vulnerabilities place protocol funds immediately at risk.

| Framework | Immutability | Autonomous Execution | Financial Harm | Blockchain-Specific | Empirical Validation |
|-----------|:---:|:---:|:---:|:---:|:---:|
| CVSS v3.1 | No | No | No | No | No |
| BVSS (Halborn) | Partial | No | Partial | Partial | No |
| Ahmad et al. (2021) | No | No | Partial | No | Partial |
| **SVTRAF** | **Yes** | **Yes** | **Yes** | **Yes** | **Yes** |

**Gap:** No existing framework combines blockchain-specific design with rigorous
empirical validation against real-world exploit data.

In [ ]:
# Define the SVTRAF vulnerability taxonomy: 4 categories, 17 vulnerabilities
taxonomy = {
    'Coding Errors': {
        'description': 'Implementation mistakes in contract code',
        'vulnerabilities': [
            {'name': 'Reentrancy', 'cwe': 'CWE-841', 'swc': 'SWC-107'},
            {'name': 'Integer Overflow/Underflow', 'cwe': 'CWE-190', 'swc': 'SWC-101'},
            {'name': 'Unchecked Return Values', 'cwe': 'CWE-252', 'swc': 'SWC-104'},
            {'name': 'Uninitialized Storage Pointers', 'cwe': 'CWE-824', 'swc': 'SWC-109'},
            {'name': 'Delegatecall to Untrusted Contract', 'cwe': 'CWE-829', 'swc': 'SWC-112'},
        ]},
    'Logical Flaws': {
        'description': 'Design errors in contract business logic',
        'vulnerabilities': [
            {'name': 'Timestamp Dependence', 'cwe': 'CWE-829', 'swc': 'SWC-116'},
            {'name': 'State Machine Failure', 'cwe': 'CWE-662', 'swc': 'N/A'},
            {'name': 'Faulty Business Logic', 'cwe': 'CWE-840', 'swc': 'N/A'},
            {'name': 'Inconsistent State Updates', 'cwe': 'CWE-662', 'swc': 'N/A'},
        ]},
    'Access Control Weaknesses': {
        'description': 'Permission and authorisation failures',
        'vulnerabilities': [
            {'name': 'Missing Function Access Control', 'cwe': 'CWE-284', 'swc': 'SWC-105'},
            {'name': 'tx.origin Authentication', 'cwe': 'CWE-477', 'swc': 'SWC-115'},
            {'name': 'Incorrect Inheritance Order', 'cwe': 'CWE-696', 'swc': 'SWC-125'},
            {'name': 'Unprotected Selfdestruct', 'cwe': 'CWE-284', 'swc': 'SWC-106'},
        ]},
    'Economic Design Flaws': {
        'description': 'Vulnerabilities in protocol economics and incentive design',
        'vulnerabilities': [
            {'name': 'Flash Loan Attack Vector', 'cwe': 'CWE-841', 'swc': 'N/A'},
            {'name': 'Price Oracle Manipulation', 'cwe': 'CWE-345', 'swc': 'N/A'},
            {'name': 'Front-Running / MEV', 'cwe': 'CWE-362', 'swc': 'SWC-114'},
            {'name': 'Liquidation Cascade', 'cwe': 'CWE-840', 'swc': 'N/A'},
        ]},
}

total = 0
for cat, d in taxonomy.items():
    print(f"\n{cat}  —  {d['description']}")
    print('-' * 75)
    for v in d['vulnerabilities']:
        print(f"  • {v['name']:<38} {v['cwe']:<10} {v['swc']}")
    total += len(d['vulnerabilities'])
print(f"\nTOTAL: {total} vulnerabilities across {len(taxonomy)} categories")

## 2. Composite Scoring Formula (v2)

$$\text{SVTRAF} = 10 \times [\,0.40 \cdot ISC + 0.20 \cdot ESC + 0.25 \cdot AMP + 0.15 \cdot INT\,]$$

where (inputs normalised to $[0,1]$):

- $ISC = 1 - (1-f)(1-0.5s)$ — saturating **Impact Sub-Score** combining Financial Harm ($f$) and Scope ($s$)
- $ESC = e$ — **Exploitability Sub-Score**
- $AMP = 0.6i + 0.4t$ — blockchain **Amplifier** combining Immutability ($i$) and Transaction Automation ($t$)
- $INT = ISC \times AMP$ — **Interaction term**: impact is amplified when the flaw is both unpatchable and autonomous

### Weight justification (literature-grounded)

| Dimension | Weight | Justification |
|---|---|---|
| Financial Harm (in ISC, 0.40 block) | Highest | Darvishi et al. (2024): financial loss is the strongest real-world severity indicator (r = 0.89) |
| Immutability (0.6 of AMP) | High | Zhou et al. (2023): unpatchable flaws persist; immutability materially raises effective severity |
| Automation (0.4 of AMP) | High | Chaliasos et al. (2024): the large majority of DeFi exploits execute autonomously |
| Exploitability (0.20) | Medium | Standard risk model — likelihood dimension (impact × likelihood) |
| Scope (inside ISC, saturating) | Lower | Secondary blast-radius effect with diminishing returns |

In [ ]:
def score_svtraf_composite(f, e, i, t, s):
    """
    Calculate the SVTRAF composite score (v2) for a vulnerability.

    Parameters (all on a 0-10 scale, internally normalised to 0-1):
        f : Financial Harm
        e : Exploitability
        i : Immutability Impact
        t : Transaction Automation
        s : Scope

    Returns: SVTRAF score on a 0-10 scale.
    """
    # Normalise inputs to 0-1
    f, e, i, t, s = f/10, e/10, i/10, t/10, s/10

    isc = 1 - (1 - f) * (1 - 0.5 * s)   # Impact sub-score (saturating)
    esc = e                              # Exploitability sub-score
    amp = 0.6 * i + 0.4 * t              # Blockchain amplifier
    int_term = isc * amp                 # Interaction term

    score = 10 * (0.40 * isc + 0.20 * esc + 0.25 * amp + 0.15 * int_term)
    return round(score, 4)

# Sanity tests across the severity spectrum
tests = [
    ('Patchable, manual, low impact',      2, 3,  0,  0, 2),
    ('Immutable, manual, medium impact',   6, 4, 10,  0, 5),
    ('Patchable, automated, medium',       6, 8,  0, 10, 5),
    ('Immutable, automated, high impact',  9, 8, 10, 10, 8),
]

print(f"{'Scenario':<38}{'Score':>8}   Severity")
print('-' * 62)
for name, f, e, i, t, s in tests:
    sc = score_svtraf_composite(f, e, i, t, s)
    sev = 'CRITICAL' if sc >= 8 else 'HIGH' if sc >= 6 else 'MEDIUM' if sc >= 4 else 'LOW'
    print(f'{name:<38}{sc:>8.2f}   {sev}')

## Summary

- **Gap identified:** existing frameworks ignore immutability, autonomous execution, and financial harm.
- **Taxonomy defined:** 4 root-cause categories, 17 vulnerabilities, mapped to CWE and SWC.
- **Formula defined and tested:** the composite v2 formula produces the expected severity ordering — an immutable, automated, high-impact flaw scores CRITICAL while a patchable, manual, low-impact flaw scores LOW.

**Next:** Notebook 2 — Dataset Preparation (150 stratified smart contracts).